# FINANCE 361 — Topic 12: Return Predictability
## Sorted Portfolio Methodology — Interactive Python Notebook

**University of Auckland | Dr. Zicheng (Leo) Xiao**

---

This notebook implements the **7-step sorted portfolio methodology** from Topic 12, using toy data and then real Fama-French factor data.

### Learning Objectives
1. Build a panel dataset and lag a characteristic correctly
2. Sort stocks into quintile portfolios within each month
3. Compute time-series of portfolio returns and the 5M1 factor portfolio
4. Run a Fama-French 5-factor regression with Newey-West standard errors
5. Interpret alpha and factor loadings


## Setup — Install & Import

In [ ]:
# Install any missing packages
!pip install pandas numpy matplotlib seaborn statsmodels pandas-datareader --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import statsmodels.api as sm
from IPython.display import display

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)
print('All packages loaded successfully.')

---
## Step 1 — Build a Panel Dataset

Our toy dataset has:
- **15 stocks** (S01 – S15)
- **24 months** (2 years of monthly data)
- Two variables per stock-month: `ret` (monthly return %) and `x` (a mystery characteristic)

The characteristic `x` is constructed to have a genuine positive relationship with **next-month** returns, with noise added.


In [ ]:
# ── Generate toy panel data ────────────────────────────────────────────────
n_stocks = 15
n_months = 24

stocks = [f'S{i:02d}' for i in range(1, n_stocks + 1)]
dates  = pd.date_range('2020-01', periods=n_months, freq='MS')

rows = []
for s in stocks:
    # Each stock has a persistent component to x
    x_series = np.random.uniform(1, 9, n_months)
    # Returns are driven by lagged x + noise
    ret_series = 0.5 * np.roll(x_series, 1) + np.random.normal(0, 3, n_months)
    ret_series[0] = np.nan  # first month has no lagged x
    for t in range(n_months):
        rows.append({'stock': s, 'date': dates[t],
                     'ret': ret_series[t], 'x': x_series[t]})

panel = pd.DataFrame(rows)
panel = panel.sort_values(['stock', 'date']).reset_index(drop=True)

print(f'Panel shape: {panel.shape}')
print(f'Stocks: {n_stocks}, Months: {n_months}')
print()
display(panel.pivot(index='date', columns='stock', values='ret').round(2).head(6))

---
## Step 2 — Lag the Characteristic

We must use **prior-month** x to predict **this-month** return.  
Using same-period x creates a **look-ahead bias** — we'd be using information we couldn't have had in real time.


In [ ]:
# Lag x by one month within each stock
panel = panel.sort_values(['stock', 'date'])
panel['x_lag1'] = panel.groupby('stock')['x'].shift(1)

# Drop rows with missing lagged x or missing returns
panel_clean = panel.dropna(subset=['x_lag1', 'ret']).copy()

print(f'Rows after lagging and dropping NaN: {len(panel_clean)}')
print()
print('First 8 rows for stock S01:')
display(panel_clean[panel_clean['stock'] == 'S01'][['date','ret','x','x_lag1']].head(8))

---
## Step 3 — Assign NYSE-Style Quintiles

In each month, we rank stocks by their **lagged x** and assign them to quintiles (Q1 = lowest x, Q5 = highest x).

> **Tech Note:** In real research, breakpoints are computed using only NYSE-listed stocks to avoid microcap contamination. Here we use all 15 stocks since the sample is too small to split.


In [ ]:
def assign_quintile(s):
    """Assign quintile labels 1-5 based on ranked values within a group."""
    return pd.qcut(s, q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')

panel_clean['quintile'] = (
    panel_clean.groupby('date')['x_lag1']
    .transform(assign_quintile)
    .astype(int)
)

print('Quintile distribution (should be ~3 stocks per quintile per month):')
display(
    panel_clean.groupby(['date','quintile'])
    .size().unstack('quintile').rename_axis('').head(6)
)

---
## Steps 4 & 5 — Portfolio Returns & Mean Returns

Compute the **equal-weighted mean return** within each quintile for each month, then average over time.

> **Tech Note:** Real research uses **value-weighted** returns (weighted by market cap). We use equal-weighting here for simplicity.


In [ ]:
# Equal-weighted portfolio returns per quintile per month
qret = (
    panel_clean.groupby(['date', 'quintile'])['ret']
    .mean()
    .unstack('quintile')
)
qret.columns = [f'Q{i}' for i in range(1, 6)]

# Factor portfolio: Long Q5, Short Q1
qret['5M1'] = qret['Q5'] - qret['Q1']

print('Time-series of quintile portfolio returns (first 6 months):')
display(qret.round(2).head(6))

print()
print('=== MEAN RETURNS BY QUINTILE ===')
means = qret.mean()
display(means.to_frame('Mean Monthly Return (%)').round(4))

---
## Visualisation 1 — Quintile Return Bar Chart

If x is a positive predictor, we should see a **monotone increasing** pattern from Q1 to Q5.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: Quintile mean returns ────────────────────────────────────────────
q_labels = ['Q1\n(Low x)', 'Q2', 'Q3', 'Q4', 'Q5\n(High x)']
q_means  = [means[f'Q{i}'] for i in range(1, 6)]
colors   = ['#ef4444', '#f97316', '#eab308', '#22c55e', '#2563eb']

bars = axes[0].bar(q_labels, q_means, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Mean Monthly Return by Quintile', fontweight='bold')
axes[0].set_ylabel('Mean Return (%)')
axes[0].axhline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, q_means):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f}%', ha='center', va='bottom', fontsize=9)

# ── Right: 5M1 time-series ─────────────────────────────────────────────────
ts_colors = ['#2563eb' if v >= 0 else '#dc2626' for v in qret['5M1']]
axes[1].bar(range(len(qret)), qret['5M1'], color=ts_colors, edgecolor='white', linewidth=0.5)
axes[1].axhline(means['5M1'], color='#16a34a', linestyle='--', linewidth=1.5,
                label=f'Average = {means["5M1"]:.2f}%')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('5M1 Factor Portfolio — Monthly Returns', fontweight='bold')
axes[1].set_ylabel('Return (%)')
axes[1].set_xlabel('Month')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('topic12_quintile_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'5M1 average return: {means["5M1"]:.2f}% per month  →  {(1+means["5M1"]/100)**12 - 1:.1%} annualised')

---
## Step 6 — Fama-French 5-Factor Regression

We download the **Fama-French 5 factors** from Ken French's data library and regress the 5M1 portfolio on them.  
A significant **alpha (α)** means the strategy earns risk-adjusted abnormal returns.

$$\text{5M1}_t = \alpha + \beta_{\text{MKT}} \cdot \text{MKTRF}_t + \beta_{\text{SMB}} \cdot \text{SMB}_t + \beta_{\text{HML}} \cdot \text{HML}_t + \beta_{\text{CMA}} \cdot \text{CMA}_t + \beta_{\text{RMW}} \cdot \text{RMW}_t + \varepsilon_t$$


In [ ]:
# Download Fama-French 5 factors from Ken French's website
import pandas_datareader.data as web

try:
    ff5_raw = web.DataReader('F-F_Research_Data_5_Factors_2x3',
                             'famafrench', start='2019-01', end='2022-12')[0]
    ff5 = ff5_raw / 100  # convert from percent to decimal
    ff5.index = pd.to_datetime(ff5.index.to_timestamp())
    ff5.index.name = 'date'
    print('FF5 factors downloaded from Ken French website.')
    display(ff5.head(3))
except Exception as e:
    print(f'Download failed: {e}')
    print('Generating simulated FF5 factors for the regression...')
    # Simulate realistic FF5 factors
    np.random.seed(99)
    ff5 = pd.DataFrame({
        'Mkt-RF': np.random.normal(0.006, 0.045, n_months),
        'SMB':    np.random.normal(0.002, 0.025, n_months),
        'HML':    np.random.normal(0.003, 0.027, n_months),
        'RMW':    np.random.normal(0.003, 0.020, n_months),
        'CMA':    np.random.normal(0.002, 0.018, n_months),
        'RF':     np.random.uniform(0.0000, 0.0004, n_months),
    }, index=dates)
    ff5.index.name = 'date'
    display(ff5.head(3))

In [ ]:
# Merge factor portfolio with FF5 factors
ret_5m1 = qret[['5M1']].copy()
ret_5m1.index = pd.to_datetime(ret_5m1.index)
ret_5m1['5M1'] = ret_5m1['5M1'] / 100  # convert to decimal

ff5.index = pd.to_datetime(ff5.index)

data = ret_5m1.join(ff5, how='inner').dropna()

# Dependent variable: 5M1 excess return
y = data['5M1']

# Independent variables: FF5 factors
factor_cols = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
X = sm.add_constant(data[factor_cols])

# OLS with Newey-West HAC standard errors (6 lags)
model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 6})

print(model.summary())

---
## Step 7 — Interpret the Results


In [ ]:
alpha       = model.params['const']
t_alpha     = model.tvalues['const']
p_alpha     = model.pvalues['const']
r_squared   = model.rsquared
n_obs       = int(model.nobs)
alpha_annual = (1 + alpha) ** 12 - 1

print('=' * 55)
print('  SORTED PORTFOLIO RESULTS SUMMARY')
print('=' * 55)
print(f'  Alpha (monthly):   {alpha*100:>8.4f}%')
print(f'  Alpha (annual):    {alpha_annual*100:>8.2f}%')
print(f'  t-statistic:       {t_alpha:>8.2f}')
print(f'  p-value:           {p_alpha:>8.4f}')
print(f'  R²:                {r_squared:>8.4f}')
print(f'  N (months):        {n_obs:>8d}')
print('=' * 55)

if abs(t_alpha) > 1.96:
    print(f'\n✅ Alpha is statistically significant (|t| = {abs(t_alpha):.2f} > 1.96)')
    print('   → Evidence of return predictability after risk adjustment.')
else:
    print(f'\n❌ Alpha is NOT significant (|t| = {abs(t_alpha):.2f} < 1.96)')
    print('   → No evidence of return predictability after risk adjustment.')

print()
print('Factor loadings (beta coefficients):')
for f in factor_cols:
    sig = '✓ sig' if abs(model.tvalues[f]) > 1.96 else '  n.s.'
    print(f'  {f:<8}: β = {model.params[f]:>7.4f}  t = {model.tvalues[f]:>5.2f}  {sig}')

---
## Bonus: Visualise the Full Regression Results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: Coefficient plot ─────────────────────────────────────────────────
params  = model.params
ci_low  = model.conf_int()[0]
ci_high = model.conf_int()[1]
names   = ['α (Intercept)'] + factor_cols

y_pos = range(len(names))
axes[0].errorbar(
    params.values, y_pos,
    xerr=[(params - ci_low).values, (ci_high - params).values],
    fmt='o', color='#2563eb', ecolor='#93c5fd', capsize=5, markersize=7
)
axes[0].axvline(0, color='#dc2626', linestyle='--', linewidth=1)
axes[0].set_yticks(list(y_pos))
axes[0].set_yticklabels(names)
axes[0].set_xlabel('Coefficient (95% CI)')
axes[0].set_title('Factor Regression Coefficients\n(5M1 on FF5 Factors)', fontweight='bold')

# ── Right: Actual vs fitted ────────────────────────────────────────────────
axes[1].scatter(model.fittedvalues * 100, y * 100, alpha=0.7, color='#2563eb', s=50)
lo, hi = (y * 100).min(), (y * 100).max()
axes[1].plot([lo, hi], [lo, hi], 'r--', linewidth=1)
axes[1].set_xlabel('Fitted values (%)')
axes[1].set_ylabel('Actual 5M1 return (%)')
axes[1].set_title(f'Actual vs Fitted Returns\n(R² = {r_squared:.3f})', fontweight='bold')

plt.tight_layout()
plt.savefig('topic12_regression.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🚀 Challenge: Try Your Own Characteristic

Modify the cell below to change the signal strength and see how it affects the results.


In [ ]:
# ── EXPERIMENT: Change signal_strength (0 = no signal, 1 = strong signal) ─
signal_strength = 0.5   # ← try values between 0 and 1

np.random.seed(42)
rows_exp = []
for s in stocks:
    x_s = np.random.uniform(1, 9, n_months)
    r_s = signal_strength * np.roll(x_s, 1) + np.random.normal(0, 3, n_months)
    r_s[0] = np.nan
    for t in range(n_months):
        rows_exp.append({'stock': s, 'date': dates[t], 'ret': r_s[t], 'x': x_s[t]})

panel_exp = pd.DataFrame(rows_exp)
panel_exp['x_lag1'] = panel_exp.groupby('stock')['x'].shift(1)
panel_exp = panel_exp.dropna()
panel_exp['quintile'] = panel_exp.groupby('date')['x_lag1'].transform(assign_quintile).astype(int)

qret_exp = panel_exp.groupby(['date','quintile'])['ret'].mean().unstack().rename(columns=lambda c: f'Q{c}')
qret_exp['5M1'] = qret_exp['Q5'] - qret_exp['Q1']

means_exp = qret_exp.mean()
fig, ax = plt.subplots(figsize=(7, 4))
q_labels = ['Q1\n(Low x)', 'Q2', 'Q3', 'Q4', 'Q5\n(High x)']
bars = ax.bar(q_labels, [means_exp[f'Q{i}'] for i in range(1,6)],
              color=['#ef4444','#f97316','#eab308','#22c55e','#2563eb'])
ax.set_title(f'Quintile Returns | Signal Strength = {signal_strength}', fontweight='bold')
ax.set_ylabel('Mean Monthly Return (%)')
ax.axhline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, [means_exp[f'Q{i}'] for i in range(1,6)]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05 if val>=0 else bar.get_height()-0.3,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print(f'5M1 average: {means_exp["5M1"]:.2f}% per month')

---
## Summary

| Step | What we did |
|------|-------------|
| 1 | Built a panel dataset (stocks × months) |
| 2 | Lagged characteristic x by one period (avoid look-ahead bias) |
| 3 | Sorted stocks into quintiles within each month |
| 4 | Computed equal-weighted portfolio returns per quintile |
| 5 | Formed 5M1 = Q5 − Q1 (the factor/hedge portfolio) |
| 6 | Regressed 5M1 on Fama-French 5 factors (Newey-West SE) |
| 7 | Tested whether α ≠ 0 (evidence of return predictability) |

**A significant positive alpha means x predicts returns beyond what known risk factors can explain.**

---
*FINANCE 361 | University of Auckland | Dr. Zicheng (Leo) Xiao*
